# Environment check

## Verify NPU health

In [1]:
!npu-smi info

+--------------------------------------------------------------------------------------------------------+
| npu-smi 23.0.0                                   Version: 23.0.0                                       |
+-------------------------------+-----------------+------------------------------------------------------+
| NPU     Name                  | Health          | Power(W)     Temp(C)           Hugepages-Usage(page) |
| Chip    Device                | Bus-Id          | AICore(%)    Memory-Usage(MB)                        |
+===============================+=================+======================================================+
| 0       310B1                 | Alarm           | 0.0          51                15    / 15            |
| 0       0                     | NA              | 0            3360 / 23673                            |
+===============================+=================+======================================================+


The NPU health is in state "Alarm". Note that this is expected on the OrangePi AIpro \(20T\). The development board does not implement the current detection function of standard Ascend 310B NPUs which is documented in the [Ascend forum](https://www.hiascend.com/forum/thread-0265146668062280052-1-1.html).

In [2]:
!npu-smi info -t health -i 0 -c 0

	Health Status                  : Alarm
	Error Code                     : 80E3A203 80F18003
	Error Information              : node type=LPM, sensor type=Chip Hardware, event state=The function of obtaining the current is abnormal

	                               : node type=DDRA, sensor type=RAS State, event state=internal config error


The following error codes are expected:

1. `80E3A203`: the current detection function is not implemented
1. `80F18003`: reported by users on some setups, does not affect model training and inference

Any other error code may be a hardware-related issue and it is recommended to post your enquiry in the Ascend forum.

## Verify CANN toolkit and kernels versions

In [3]:
!cat /usr/local/Ascend/ascend-toolkit/latest/aarch64-linux/ascend_toolkit_install.info

package_name=Ascend-cann-toolkit
version=8.0.0
innerversion=V100R001C20SPC001B251
compatible_version=[V100R001C15],[V100R001C17],[V100R001C18],[V100R001C19],[V100R001C20]
arch=aarch64
os=linux
path=/usr/local/Ascend/ascend-toolkit/8.0.0/aarch64-linux


CANN toolkit version `8.0.0` is installed. Your installed version may by different.

In [4]:
!cat /usr/local/Ascend/ascend-toolkit/latest/opp/version.info

Version=7.6.0.1.220
version_dir=8.0.0
timestamp=20241231_000821303
required_package_amct_acl_version="7.6"
required_package_aoe_version="7.6"
required_package_compiler_version="7.6"
required_package_fwkplugin_version="7.6"
required_package_hccl_version="7.6"
required_package_nca_version="7.6"
required_package_ncs_version="7.6"
required_package_opp_kernel_version="7.6"
required_package_runtime_version="7.6"
required_package_toolkit_version="7.6"


The CANN kernels version is `7.6.0.1.220`.

## Verify MindSpore version and successful installation

In [5]:
%pip show mindspore

Name: mindspore
Version: 2.4.10
Summary: MindSpore is a new open source deep learning training/inference framework that could be used for mobile, edge and cloud scenarios.
Home-page: https://www.mindspore.cn
Author: The MindSpore Authors
Author-email: contact@mindspore.cn
License: Apache 2.0
Location: /home/HwHiAiUser/.local/lib/python3.9/site-packages
Requires: asttokens, astunparse, numpy, packaging, pillow, protobuf, psutil, safetensors, scipy
Required-by: mindnlp
Note: you may need to restart the kernel to use updated packages.


MindSpore `2.4.10` is installed. Let's verify whether it is installed successfully.

In [6]:
import mindspore
mindspore.set_context(device_target='Ascend')
mindspore.run_check()

/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/miniconda3/lib/python3.9/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore version:  2.4.10


[WARNING] GE_ADPT(11167,e7fef199f120,python):2026-03-15-14:44:01.999.265 [mindspore/ccsrc/transform/acl_ir/op_api_exec.cc:141] GetAscendDefaultCustomPath] Checking whether the so exists or if permission to access it is available: /usr/local/Ascend/ascend-toolkit/latest/opp/vendors/customize_vision/op_api/lib/libcust_opapi.so


The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


It is normal to see a few warning messages. As long as the following line of output appears, MindSpore is installed successfully and functional.

```text
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!
```

### `te` Module initialization failure

If the last line of output does not match and you see error messages about Python modules being uninitialized, it's likely because of overly restrictive permissions in the CANN kernels directory `/usr/local/Ascend/ascend-toolkit/latest/opp/vendors`.

In [7]:
!ls -l /usr/local/Ascend/ascend-toolkit/latest/opp/vendors

total 8
-rwxr-x---+ 1 root root   31 Sep  9  2025 config.ini
drwxr-x---+ 5 root root 4096 Sep  9  2025 customize_vision


Items in this directory are owned by the `root` user and group. Furthermore, non-root users such as `HwHiAiUser` are not granted read access.

Without granting overly permissive file access or adding the `HwHiAiUser` to the `root` group, we can leverage POSIX ACLs to specifically grant the `HwHiAiUser`:

1. Read access to all files recursively under the CANN kernels directory
1. Read and execute access to all directories recursively under the CANN kernels directory

The approach is described in this Ascend forum [post](https://www.hiascend.com/forum/thread-02144208449043302121-1-1.html).

```bash
sudo find /usr/local/Ascend/ascend-toolkit/latest/opp/vendors \

    -type f \

    -exec setfacl -m u:HwHiAiUser:r {} +

sudo find /usr/local/Ascend/ascend-toolkit/latest/opp/vendors \

    -type d \

    -exec setfacl -m u:HwHiAiUser:rx {} +
```

Once the file and directory permissions are fixed, re-run the MindSpore installation check.

In [8]:
mindspore.run_check()

MindSpore version:  2.4.10
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!
